In [1]:
import pandas as pd
import numpy as np
import torch.nn as nn
from model import LSTMv1
from torch.utils import data
import torch
from sklearn.model_selection import train_test_split

In [2]:
noms = pd.read_csv("communes-france-2026.csv", dtype=str)["nom_standard"]
noms = noms.to_list()
for i, nom in enumerate(noms):
    noms[i] = nom.lower()

In [3]:
listVocab = [] # liste du vocabulaire, ici le vocabulaire = les lettres possibles
for nom in noms:
    for letter in nom:
        if letter not in listVocab:
            listVocab.append(letter)
i = 0
listVocab.sort()
while listVocab[i] != 'a':
    i += 1
listVocab[:-i], listVocab[-i:] = listVocab[i:], listVocab[0:i] # juste pour avoir la liste qui commence par a et qui finit par les caractères spéciaux

In [4]:
nVocab = len(listVocab)
nVocab

45

In [74]:
encodings = np.eye(nVocab) # encoding i = que des 0 sauf l'élt i qui vaut 1
dicoEncodings = {}
inverseEncodings = {}
for i, letter in enumerate(listVocab):
    dicoEncodings[letter] = encodings[i]
    inverseEncodings[i] = letter

In [6]:
rnn = LSTMv1(2, 64, nVocab)

In [7]:
def encode(listOfWords):
    encoded = []
    for word in listOfWords:
        embedding = []
        for letter in word:
            embedding.append(dicoEncodings[letter])
        encoded.append(np.array(embedding))
    return encoded

In [8]:
nomsEncoded = encode(noms)

In [9]:
torch.tensor(nomsEncoded[0])

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], dtype=torch.float64)

In [10]:
X_train, X_test = train_test_split(nomsEncoded, test_size=0.2, shuffle=True)

In [69]:
optimizer = torch.optim.Adam(rnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
nEpoch = 5
batchSize = 64
for epoch in range(nEpoch):
    total_loss = 0
    print(epoch)
    for i, x in enumerate(X_train):
        xTensor = torch.tensor(x).float()
        if xTensor.shape[0] < 2: #il y a quand même une ville qui s'appelle "y"
            continue
        target = xTensor[1:]
        target = target.argmax(dim=1)
        inputRnn = xTensor[:-1]
        pred = rnn(inputRnn)
        loss = criterion(pred, target)
        loss.backward()
        if i%batchSize == 0:
            optimizer.step()
            optimizer.zero_grad()
        if i%1000 == 0:
            print(i)
    

0
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
1
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
2
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
3
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
4
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000


In [11]:
len(X_train)

27894

In [12]:
len(X_test)

6974

In [13]:
rnn(torch.tensor(nomsEncoded[0]).float())[0].shape

torch.Size([45])

In [14]:
torch.tensor(nomsEncoded[0]).shape

torch.Size([23, 45])

In [63]:
for i in range(len(noms)):
    if len(noms[i])<2: 
        print(i)
len(noms[0])

31680


23

In [64]:
noms[31680]

'y'

In [49]:
a = torch.tensor([1, 0])
b = torch.tensor([10, 0]).float()
criterion(b.unsqueeze(0), a.unsqueeze(0))

RuntimeError: Expected floating point type for target with class probabilities, got Long

In [66]:
27894/64

435.84375

In [73]:
encodings[1].argmax()

np.int64(1)

In [149]:
start = "a" 
rnn.eval()
startEncoded = [dicoEncodings[start]]
result = start
for i in range(18):
    nextChar = rnn(torch.tensor(startEncoded).float())[-1]
    probs = torch.softmax(nextChar, dim=-1)
    nextChar = torch.multinomial(probs, num_samples=1)
    nextChar = inverseEncodings[nextChar.item()]
    startEncoded.append(dicoEncodings[nextChar])
    result += nextChar


In [150]:
result

'atrichante-sur-lieu'